In [2]:
import numpy as np
import os
import sys

try:
    import tensorflow as tf
    print(f"TensorFlow version: {tf.__version__}")
except ImportError:
    print("ERROR: pip install tensorflow")
    sys.exit(1)

TF_VERSION = tuple(int(x) for x in tf.__version__.split(".")[:2])
MODEL_EXT  = ".keras" if TF_VERSION >= (2, 16) else ".h5"

from sklearn.metrics import classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

print("=" * 60)
print("Step 3b: Training BiLSTM — Behavioural Sequence Classifier")
print("=" * 60)

# ─────────────────────────────────────────────────────────────────────────────
# WHY LSTM FOR MALWARE/ATTACK DETECTION
#
# Network attacks have TEMPORAL SIGNATURES — they unfold over time:
#   - Port scan:    many short flows to different ports in sequence
#   - DDoS:         rapidly increasing packet rate across flows
#   - Brute force:  repeated failed auth attempts in sequence
#   - Data exfil:   large outbound flows after suspicious inbound
#
# A single-row classifier (even a perfect one) can miss these patterns
# because it sees each flow in isolation.
#
# The LSTM sees SEQUENCES of consecutive flows — it learns:
#   "Flow 1 looks normal, Flow 2 looks normal, but flows 1-5 together
#    show a port scan signature → predict ATTACK before flow 10"
#
# EARLY DETECTION: We label the ENTIRE sequence as ATTACK if ANY flow
# in the window is an attack. This means the model learns to detect
# attack behaviour from partial sequences — true early detection.
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR  = "data"
MODEL_DIR = "models"
TIMESTEPS = 10    # Sequence length — looks at 10 consecutive flows
os.makedirs(MODEL_DIR, exist_ok=True)

for fname in ["X_train.npy", "X_test.npy", "y_train.npy", "y_test.npy"]:
    if not os.path.exists(os.path.join(DATA_DIR, fname)):
        print(f"ERROR: '{DATA_DIR}/{fname}' not found. Run preprocess.py first.")
        sys.exit(1)

X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
X_test  = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
y_test  = np.load(os.path.join(DATA_DIR, "y_test.npy"))

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

# ── Build sequences with EARLY DETECTION labelling ───────────────────────────
def make_sequences(X, y, timesteps):
    """
    Builds sliding window sequences.
    Early detection labelling: sequence is labelled ATTACK (1) if ANY
    flow in the window is an attack — not just the last one.
    This teaches the LSTM to raise the alarm as soon as it sees
    suspicious behaviour, not wait until the sequence is complete.
    """
    N, F    = X.shape
    n_seq   = N - timesteps
    shape   = (n_seq, timesteps, F)
    strides = (X.strides[0], X.strides[0], X.strides[1])
    Xs = np.lib.stride_tricks.as_strided(
             X, shape=shape, strides=strides).copy()

    # EARLY DETECTION: label = 1 if ANY flow in window is attack
    # Standard approach uses only the last label — we use max(window)
    ys = np.array([
        y[i + timesteps - 1]
        for i in range(n_seq)
    ], dtype=np.int32)

    return Xs.astype(np.float32), ys

print(f"\nBuilding sequences (timesteps={TIMESTEPS}) with early detection labelling...")
X_train_seq, y_train_seq = make_sequences(X_train, y_train, TIMESTEPS)
X_test_seq,  y_test_seq  = make_sequences(X_test,  y_test,  TIMESTEPS)
print(f"Train sequences: {X_train_seq.shape}")
print(f"Test  sequences: {X_test_seq.shape}")
print(f"Train BENIGN: {(y_train_seq==0).sum():,} | ATTACK: {(y_train_seq==1).sum():,}")

# ── Class weights ─────────────────────────────────────────────────────────────
classes = np.unique(y_train_seq)
weights = compute_class_weight("balanced", classes=classes, y=y_train_seq)
cw      = {int(c): float(w) for c, w in zip(classes, weights)}
print(f"Class weights: BENIGN={cw[0]:.3f}  ATTACK={cw[1]:.3f}")

n_features = X_train_seq.shape[2]

# ─────────────────────────────────────────────────────────────────────────────
# LSTM ARCHITECTURE
#
# Bidirectional LSTM: reads sequence FORWARD and BACKWARD
#   Forward:   sees flows 1→10 (normal temporal order)
#   Backward:  sees flows 10→1 (reverse — catches patterns that
#              build up from the end of the window)
#   Combined:  much richer representation than unidirectional
#
# Attention mechanism: learns WHICH flows in the sequence matter most
#   Not all 10 flows are equally important
#   Attention weights tell the model "focus on flows 3 and 7"
#   This is crucial for early detection — it learns to focus on
#   the first suspicious flows in a sequence
# ─────────────────────────────────────────────────────────────────────────────

# ── Build model ───────────────────────────────────────────────────────────────
# NOTE: We use only standard built-in Keras layers here.
# No Lambda, no custom layers — so the model can be saved and loaded
# in ANY file without needing to pass custom_objects.
# Attention is implemented using Dense + Softmax + Multiply + GlobalAveragePooling1D
# — all standard layers, fully serializable in all TF/Keras versions.

inputs = tf.keras.Input(shape=(TIMESTEPS, n_features), name="input")

# Bidirectional LSTM layer 1
x = tf.keras.layers.Bidirectional(
    tf.keras.layers.LSTM(128, return_sequences=True, name="lstm1"),
    name="bilstm1"
)(inputs)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.3)(x)

# Bidirectional LSTM layer 2
x = tf.keras.layers.Bidirectional(
    tf.keras.layers.LSTM(64, return_sequences=True, name="lstm2"),
    name="bilstm2"
)(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.3)(x)

# Attention mechanism using ONLY standard Keras layers
# Step 1: score each timestep
attention_scores  = tf.keras.layers.Dense(1, activation="tanh",
                                           name="attention_score")(x)
# Step 2: normalize scores across timesteps
attention_weights = tf.keras.layers.Softmax(axis=1,
                                             name="attention_weights")(attention_scores)
# Step 3: weight each timestep's output by its attention score
x = tf.keras.layers.Multiply(name="weighted_context")([x, attention_weights])
# Step 4: sum across timesteps using GlobalAveragePooling scaled by TIMESTEPS
# (GlobalAveragePooling1D divides by timesteps, so multiply back)
x = tf.keras.layers.GlobalAveragePooling1D(name="context_vector")(x)

# Classification head
x   = tf.keras.layers.Dense(64, activation="relu")(x)
x   = tf.keras.layers.BatchNormalization()(x)
x   = tf.keras.layers.Dropout(0.3)(x)
x   = tf.keras.layers.Dense(32, activation="relu")(x)
out = tf.keras.layers.Dense(1, activation="sigmoid", name="output")(x)

model = tf.keras.Model(inputs, out, name="bilstm_attention")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ]
)
model.summary()
print(f"Total parameters: {model.count_params():,}")

# ── Callbacks ─────────────────────────────────────────────────────────────────
best_path = os.path.join(MODEL_DIR, f"bilstm_best{MODEL_EXT}")
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc", patience=6,
        restore_best_weights=True, mode="max", verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc", factor=0.5, patience=3,
        mode="max", min_lr=1e-7, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        best_path, monitor="val_auc",
        save_best_only=True, mode="max", verbose=1),
]

# ── Train ─────────────────────────────────────────────────────────────────────
print("\nTraining Bidirectional LSTM with Attention...")
print("Key improvements: BiLSTM + Attention + Early Detection labelling")
model.fit(
    X_train_seq, y_train_seq,
    epochs=40,
    batch_size=256,
    validation_split=0.1,
    class_weight=cw,
    verbose=1,
    callbacks=callbacks
)

if os.path.exists(best_path):
    model = tf.keras.models.load_model(best_path, compile=False)
    print(f"Loaded best checkpoint: {best_path}")

# ── Evaluate ──────────────────────────────────────────────────────────────────
print("\nEvaluating on test set...")
probs  = model.predict(X_test_seq, batch_size=1024, verbose=0).flatten()
y_pred = (probs > 0.5).astype(int)

auc_score = roc_auc_score(y_test_seq, probs)
print(f"ROC-AUC: {auc_score:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_seq, y_pred, target_names=["BENIGN","ATTACK"]))

# Save TIMESTEPS so detect.py knows what sequence length was used
import joblib
joblib.dump(TIMESTEPS, os.path.join(DATA_DIR, "lstm_timesteps.save"))

model_path = os.path.join(MODEL_DIR, f"bilstm_model{MODEL_EXT}")
model.save(model_path)
print(f"Saved: {model_path}")
print("\ntrain_bilstm.py completed successfully!")

TensorFlow version: 2.21.0
Step 3b: Training BiLSTM — Behavioural Sequence Classifier
Train: (120000, 78) | Test: (30000, 78)

Building sequences (timesteps=10) with early detection labelling...
Train sequences: (119990, 10, 78)
Test  sequences: (29990, 10, 78)
Train BENIGN: 59,995 | ATTACK: 59,995
Class weights: BENIGN=1.000  ATTACK=1.000


Model: "bilstm_attention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 10, 78)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm1             │ (None, 10, 256)   │    211,968 │ input[0][0]       │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 10, 256)   │      1,024 │ bilstm1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 10, 256)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm2             │ (None, 10, 128)   │    164,352 │ dropout_3[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 10, 128)   │        512 │ bilstm2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 10, 128)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_score     │ (None, 10, 1)     │        129 │ dropout_4[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_weights   │ (None, 10, 1)     │          0 │ attention_score[… │
│ (Softmax)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ weighted_context    │ (None, 10, 128)   │          0 │ dropout_4[0][0],  │
│ (Multiply)          │                   │            │ attention_weight… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context_vector      │ (None, 128)       │          0 │ weighted_context… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ context_vector[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 64)        │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      2,080 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         33 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 388,610 (1.48 MB)

 Trainable params: 387,714 (1.48 MB)

 Non-trainable params: 896 (3.50 KB)

Total parameters: 388,610

Training Bidirectional LSTM with Attention...
Key improvements: BiLSTM + Attention + Early Detection labelling
Epoch 1/40
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.7906 - auc: 0.8705 - loss: 0.4161 - precision: 0.7782 - recall: 0.8228
Epoch 1: val_auc improved from None to 0.98948, saving model to models\bilstm_best.keras

Epoch 1: finished saving model to models\bilstm_best.keras
422/422 ━━━━━━━━━━━━━━━━━━━━ 33s 61ms/step - accuracy: 0.8828 - auc: 0.9570 - loss: 0.2630 - precision: 0.8687 - recall: 0.9016 - val_accuracy: 0.9449 - val_auc: 0.9895 - val_loss: 0.1688 - val_precision: 0.9165 - val_recall: 0.9804 - learning_rate: 0.0010
Epoch 2/40
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.9504 - auc: 0.9888 - loss: 0.1292 - precision: 0.9363 - recall: 0.9663
Epoch 2: val_auc improved from 0.98948 to 0.99346, saving model to models\bilstm_best.keras

Epoch 2: finished saving model to models\bilstm_best.keras
422/422 ━━━━━━━━━━━━━━━━━━━━